In [3]:
import os, importlib

REPO_URL = 'https://github.com/litcorp0/checkmaize.git'  # change if you fork the project

def repo_root():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            os.system(f'cd {repo} && git pull')
            return repo
        print('Repo not on this runtime yet. Cloning from GitHub...')
        result = os.system(f'git clone {REPO_URL} /content/checkmaize')
        if result != 0 or not os.path.exists(repo):
            print('Automatic clone failed. Likely causes:')
            print('  - the GitHub repo is private (make it public first), or')
            print('  - no internet on this runtime.')
            print('Manual fix - run this in a NEW cell, then re-run this cell:')
            print(f'  !git clone {REPO_URL} /content/checkmaize')
            print('Or drag the checkmaize folder into the Colab file explorer (into /content).')
            raise SystemExit
        return repo
    return os.path.abspath('..')

REPO = repo_root()
os.chdir(REPO)
print('Working in:', os.getcwd())

missing = []
for mod in ['numpy', 'PIL', 'pandas', 'yaml', 'sklearn', 'matplotlib', 'datasets', 'pytest']:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(mod)
if missing:
    print('installing missing packages:', missing)
    !pip install -q -r requirements.txt
    print('dependencies installed')
else:
    print('dependencies OK')


Repo not on this runtime yet. Cloning from GitHub...
Working in: /content/checkmaize
dependencies OK


In [4]:
import os
os.chdir(REPO)
from datasets import load_dataset
from PIL import Image
import shutil

dst = 'data/raw/plantvillage'
shutil.rmtree(dst, ignore_errors=True)
CLASS_MAP = {
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 'Cercospora_leaf_spot Gray_leaf_spot',
    'Corn_(maize)___Common_rust_': 'Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight': 'Northern_Leaf_Blight',
    'Corn_(maize)___healthy': 'healthy',
}
os.makedirs(dst, exist_ok=True)
for split in ['train', 'test']:
    # name='default' is required: the dataset README declares config 'color'
    # but the loading script only defines 'default' (which maps to color internally).
    ds = load_dataset('mohanty/PlantVillage', name='default', split=split)
    corn = [r for r in ds if r['crop'] == 'Corn (maize)']
    print(f"{split}: {len(corn)} corn rows")
    for r in corn:
        folder = CLASS_MAP[r['label']]
        out = os.path.join(dst, folder, f"{r['leaf_id']}__{r['image_path'].rsplit('/', 1)[-1]}")
        os.makedirs(os.path.dirname(out), exist_ok=True)
        r['image'].save(out)
print("plantvillage extraction done")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/6.47k [00:00<?, ?B/s]

ValueError: BuilderConfig 'color' not found. Available: ['default']

In [ ]:
import os, zipfile, shutil, glob
os.chdir(REPO)

NEEDED = {'Leaf blight': 'Leaf blight', 'Leaf spot': 'Leaf spot', 'Healthy': 'Healthy'}
NEEDED_LOWER = {k.lower(): k for k in NEEDED}

candidates = []
for name in ['Raw Data.zip', 'crop-pest-and-disease-detection.zip']:
    p = f'/content/{name}'
    if os.path.exists(p):
        candidates.append(p)
if not candidates:
    candidates = sorted(z for z in glob.glob('/content/*.zip') if 'checkmaize' not in z)
if not candidates:
    print('No dataset zip found in /content.')
    print('VS Code: drag the dataset zip (crop-pest-and-disease-detection.zip from Kaggle,')
    print('        or Raw Data.zip from Mendeley) into the file explorer, into /content.')
    print('Browser Colab: use the files pane (folder icon) upload button.')
    print('Then re-run this cell.')
    raise SystemExit
zip_path = candidates[0]
print('Using:', os.path.basename(zip_path))

extract_dir = '/content/ccmt_extract'
shutil.rmtree(extract_dir, ignore_errors=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(extract_dir)

dst = 'data/raw/ccmt_ghana'
shutil.rmtree(dst, ignore_errors=True)
os.makedirs(dst, exist_ok=True)
copied = {}

def copy_dir(src, key):
    target = os.path.join(dst, NEEDED[key])
    shutil.copytree(src, target)
    copied[key] = len(os.listdir(target))

nested_maize = None
for root, dirs, files in os.walk(extract_dir):
    if 'Maize' in dirs:
        nested_maize = os.path.join(root, 'Maize')
        break

if nested_maize and any(k.lower() in NEEDED_LOWER for k in os.listdir(nested_maize)):
    for sub in sorted(os.listdir(nested_maize)):
        if sub.lower() in NEEDED_LOWER:
            copy_dir(os.path.join(nested_maize, sub), NEEDED_LOWER[sub.lower()])
            print(sub, copied[NEEDED_LOWER[sub.lower()]])
else:
    for folder in sorted(os.listdir(extract_dir)):
        full = os.path.join(extract_dir, folder)
        if folder.startswith('Maize ') and os.path.isdir(full):
            key = folder[len('Maize '):].lower()
            if key in NEEDED_LOWER:
                copy_dir(full, NEEDED_LOWER[key])
                print(key, copied[NEEDED_LOWER[key]])

missing = [k for k in NEEDED if k not in copied]
assert not missing, f'Could not find classes {missing} in {os.path.basename(zip_path)}'
print('ccmt_ghana ready:', copied)


In [ ]:
import os
os.chdir(REPO)
!python -m data.make_manifest
!python -m data.make_splits
!python -m pytest data/tests -v


In [ ]:
import os, shutil
os.chdir(REPO)
shutil.make_archive('/content/splits', 'zip', 'data/manifests')
try:
    from google.colab import files
    files.download('/content/splits.zip')
    print('download started (browser Colab)')
except Exception:
    print('VS Code mode: the file is at /content/splits.zip')
    print('Drag it from the file explorer onto your computer, unzip it, and put the')
    print('CSV files into: checkmaize/data/manifests/')
